In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("load_type", "")
dbutils.widgets.text("merge_keys", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
load_type = dbutils.widgets.get("load_type")
merge_keys = dbutils.widgets.get("merge_keys")

In [0]:
bronze_df = spark.read.table(f"dbr_caresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

In [0]:
from pyspark.sql import functions as F
updated_df=bronze_df.withColumn("insert_timestamp",F.current_timestamp()).withColumn("updated_timestamp", F.current_timestamp())

In [0]:
from delta.tables import DeltaTable

target_table_name = f"dbr_caresync.silver.{table_name}"

if load_type == "FULL":
    updated_df.write.mode("overwrite").saveAsTable(target_table_name)

elif load_type == "APPEND":
    updated_df.write.mode("append").saveAsTable(target_table_name)

elif load_type == "MERGE":
    keys = [k.strip() for k in merge_keys.split(",")]
    merge_condition = " AND ".join([f"target.{k} = source.{k}" for k in keys])

    if spark.catalog.tableExists(target_table_name):
        target = DeltaTable.forName(spark, target_table_name)
        (
            target.alias("target")
            .merge(updated_df.alias("source"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        # Target table doesn't exist yet — fall back to full write
        updated_df.write.mode("overwrite").saveAsTable(target_table_name)

else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected FULL, APPEND, or MERGE.")

In [0]:
%skip
%sql
select * from dbr_caresync.silver.patients

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))